# 03 · Should-cost

A bottom-up cost tree per unit for every harness and connector family, priced in the producing country, landed in Brazil, and calibrated against the import price per kg that Comex Stat reports by origin.

The premises are the analyst's own, in `src/should_cost.py` and explained in `docs/assumptions.md`. They do not read the generator's true costs.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when the notebook runs from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src import should_cost as sc
from src.db import DEFAULT_DB, connect, create_schema
from src.figures import harness_calibration

con = connect(DEFAULT_DB)
create_schema(con)

copper_now = con.execute(
    "SELECT avg(copper_usd_t) FROM copper_monthly "
    "WHERE month BETWEEN DATE '2025-10-01' AND DATE '2026-09-01'"
).fetchone()[0]
print(f"Copper, average of the last 12 months used for pricing: USD {copper_now:,.0f}/t")

## Cost tree by family

Per unit at the family's reference size, produced in Brazil and in China.

In [ ]:
rows = []
for p in sc.FAMILY_PREMISES:
    for country in ("BR", "CN"):
        tree = sc.cost_tree(p, copper_now, sc.LABOR_USD_H[country])
        rows.append(
            {
                "family": p.family,
                "country": country,
                **tree,
                "landed": sc.landed(tree["ex_works"], country, p.ncm),
            }
        )
trees = pd.DataFrame(rows)
trees.round(2)

In [ ]:
harness = trees[trees["family"].str.startswith("harness")]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, country in zip(axes, ("BR", "CN"), strict=True):
    t = harness[harness["country"] == country].set_index("family")[list(sc.COMPONENTS)]
    t.index = t.index.str.replace("harness_", "").str.replace("_", " ")
    t.plot.bar(stacked=True, ax=ax, legend=False, rot=0)
    ax.set(title=f"Harness should-cost per unit, made in {country}", xlabel="", ylabel="USD")
    ax.tick_params(axis="x", labelsize=8)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=6,
    fontsize=8,
    frameon=False,
);

In [ ]:
shares = harness.assign(
    copper_share=harness["copper"] / harness["ex_works"],
    labor_share=(harness["labor"] + harness["overhead"]) / harness["ex_works"],
    duty_and_freight=harness["landed"] / harness["ex_works"] - 1,
)[["family", "country", "copper_share", "labor_share", "duty_and_freight"]]
shares.round(3)

## Calibration against Comex Stat

The harness should-cost per kg, weighted by units per vehicle, against the import price per kg of each origin in the latest full year. Mass per unit assumes copper is 60% of harness weight.

In [ ]:
calib = harness_calibration(con)
print("Year:", calib.attrs["year"])
calib.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
c = calib.sort_values("import_usd_m", ascending=False)
ax.bar(
    c["origin"],
    c["ratio"],
    color=["tab:green" if 0.85 <= r <= 1.15 else "tab:grey" for r in c["ratio"]],
)
ax.axhspan(0.85, 1.15, color="tab:green", alpha=0.1)
ax.axhline(1, color="black", lw=0.8)
ax.set(
    ylabel="should-cost / import price", title="Calibration ratio by origin (green: within ±15%)"
)
ax.tick_params(axis="x", labelsize=8);

## Reading

- The cost tree lands within 3% of the import price per kg for **Paraguay** (the largest origin, almost half of harness imports), **Thailand** and **Germany**, and within 12% for **Japan**. That is the main evidence that the labour, material and margin premises are in the right range for both low-cost and high-wage producers.
- **China** and **India** import about 27% above the model (ratio 0.78–0.79). Either their harnesses carry more content per kg than the 60% copper share implies, or the premises miss a cost; the gap is worth a closer look before negotiating with Asian suppliers.
- **Mexico**, the **United States** and **Argentina** sit at 0.4–0.6. They ship small volumes, likely specialised or complex harnesses, so they were not used to tune the premises.
- **Copper** is 27–34% of a harness at current prices; **labour plus overhead** is 14–26% in Brazil and less in China. **Freight and duty** add 23% to a Chinese harness landed in Brazil, against 3% for a duty-free Paraguayan one, which helps explain why Paraguay dominates harness imports.
- **Connectors** are not calibrated: Comex Stat unit prices for the same NCM range from USD 0.35 to USD 28 across origins, because origins count units differently.

Next: compare each supplier's prices and quotes with its should-cost (the gap analysis).